# 05 · Standard CUB70 CBM: observational test of context-dependent concepts

**Report question.** On real bird photographs, do raw concept scores depend
on the visibility of the named region and on species context after exact
concept identity is held fixed?

**Causal boundary.** CUB has no accepted clean donor-part replacement.
Therefore this notebook cannot reproduce the FunnyBird donor/source
backwash predicate. It tests converging or contrary observational evidence:
natural visibility, hidden-context scores, matched recall/raw-score gaps,
and within-concept species effects.

**Population.** Standard non-RL CUB70 CBM, seed 1, epoch 100. Full-CUB CBM
is used only as a clearly labelled same-image robustness guard.


## What this notebook can establish, and how it approaches the FunnyBird question

The broad research question is the same: does a concept score depend only on its
named region, or does surrounding species/body context help predict it? The
strongest FunnyBird result cannot be copied mechanically because CUB has no
renderer that replaces one part while holding the rest of the photograph fixed.
Therefore this notebook does **not** invent a CUB donor/source margin.

The CUB conclusion must instead be assembled from explicitly observational
predicates, in this order:

1. the available photograph, species, concept, and mask populations are known;
2. species/concept structure makes contextual shortcuts available;
3. positive labels and mapped-mask visibility sometimes disagree;
4. each exact concept output is healthy enough to interpret;
5. natural visibility of the mapped region changes raw concept `z`;
6. positive and negative labels remain separable in `z` when the mapped region
   is absent (`context_gap > 0`);
7. species still organizes `z` after exact concept and mask state are held fixed;
8. measured visibility, conflict, difficulty, support, and species account for
   some—but not necessarily all—held-out variation.

### The same three contributors, with CUB-valid substitutions

1. **visibility/occlusion** uses the released mapped mask, its area, and
   bilateral alternatives; this is a natural-image comparison, not a swap;
2. **label–visibility conflict** is the positive-label/mapped-mask-absent rate
   for every exact concept, retaining coverage counts; and
3. **exact-value difficulty** uses raw-logit health, recall, exact-value support,
   number of alternatives, and species support.

Species/body context is then tested as the remaining observational organizer.
The recall analysis restores the original standard-CBM CUB question while
borrowing only the matching and bootstrap refinements from `mcbm_recallv4`;
no MCBM numerical result is imported here.

| FunnyBird scientific question | CUB operation | Figure(s) | Claim boundary |
|---|---|---|---|
| Are the data/model outputs usable? | inventory, masks, conflict, raw-`z` health | 1–4 | same health question |
| Is species context available? | label structure and held-out species decoding | 2, 4b | availability, not causal use |
| Do named-region pixels matter? | compare naturally visible and hidden positive-labelled photographs | 5, 7 | weaker than a same-image swap |
| Does context retain concept information? | hidden-positive minus hidden-negative raw `z` | 6 | contextual prediction, not donor/source backwash |
| Are scores species-dependent? | matched recall/raw-`z` gaps and within-concept species residuals | 8, 10 | observational species association |
| What proposed contributors organize the result? | concept- and row-level held-out accounting | 9, 11 | prediction, not causal subtraction |
| Could masks be misleading? | rule-selected photographs with all masks | 12 | separates true occlusion from annotation limits |
| Does the pattern depend on CUB70 training? | same-image full-CUB guard | 12b | robustness check |

### CUB capabilities and drawbacks used in the design

CUB provides 112 exact labelled concepts, species labels, real photographs, and
11 released anatomical masks. It permits raw-score health, natural visibility,
area, bilateral-mask, recall, species, and support analyses. Its masks are
coarser than many named attributes and can be absent even when a human can see
the region. It has no accepted clean deletion or donor swap. Consequently:

- `visibility_effect` asks whether visible positive-labelled photographs score
  differently from hidden positive-labelled photographs;
- `context_gap` asks whether context distinguishes positive from negative labels
  when the mapped region is absent;
- neither quantity is the FunnyBird final margin;
- photographs and mask examples must be inspected before interpreting extremes;
- converging results support context-dependent prediction, but only FunnyBird's
  controlled swap establishes the exact backwash event.

### Predictions stated before the results

- Healthy outputs should have nonzero raw-`z` spread, positive label separation,
  and above-chance balanced accuracy/recall.
- If local visibility helps, `visibility_effect` should usually be positive and
  larger visible areas should usually accompany higher `z`.
- If context predicts the concept without the mapped region, `context_gap`
  should remain positive and species should explain held-out variation within
  exact concept and mask state.
- If conflict/support/number of alternatives are sufficient explanations, adding
  them should lower held-out concept-level error. If not, a residual remains.
- Mixed or negative visibility effects must be investigated as pose, mask
  quality, collapse, or composition before being called evidence for backwash.


## The implemented CBM and the notation used below

For image `i`, the encoder produces a latent concept vector `h_i` with one
slot for each of the `J` exact concepts.  The learned concept head for slot `j`
turns `h_ij` into the raw concept logit `z_ij`:

```text
x_i → image encoder → h_i = (h_i1, …, h_iJ)
                          ├→ learned head q_j(h_ij) → z_ij → sigmoid → p_ij
                          └→ class head on complete h_i       → species prediction
```

The implementation trains with

`L_CBM = L_task + beta × L_concept`.

The class head reads the complete latent vector `h_i`; it does not read a list of
hard 0/1 concept decisions. Thus species loss can shape the same latent slots
that the concept heads read. In these runs each concept head is a learned
`1 → 3 → 1` network, not the identity. The setup cell replays the saved head
weights on saved `h_i` and verifies that `sigmoid(z_ij)` exactly reproduces
the saved probability.

| Symbol | Meaning |
|---|---|
| `x_i` | image `i` |
| `y_i` | species label |
| `c_ij` | processed 0/1 label for exact concept `j` |
| `h_ij` | encoder's latent slot for concept `j`; also read by the class head |
| `z_ij = q_j(h_ij)` | raw concept logit after the learned head; primary grounding quantity |
| `p_ij = sigmoid(z_ij)` | bounded probability; used only for thresholded performance |
| `c_hat_ij = 1[z_ij>0]` | predicted concept presence |
| `v_ig` | whether mapped part mask `g` is visible |
| `a_ig` | visible area of mask `g` |

`L_task` is the species-classification loss. `L_concept` is the sum of the
per-concept label losses. `beta` controls their relative weight. No later plot
uses the encoder slot `h_ij` while calling it a concept logit: grounding plots
use the post-head raw score `z_ij`.

Ordinary accuracy and recall answer whether predictions agree with labels. They
do **not** answer whether the prediction came from the named pixels.


In [ ]:
import os, sys, hashlib, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

CURATED=Path(os.environ["CURATED_DATA"]); CWD=Path.cwd()
REPO=CWD if (CWD/"analysis").is_dir() else CWD.parent
sys.path.insert(0,str(REPO/"data"/"cub70"))
from cub70_parts import CUB70_PARTS, ATTRIBUTE_TYPE_TO_MASK
from relabel_cub_with_cub70 import coarse_visibility
COLORS={"head":"#56B4E9","eye":"#CC79A7","beak":"#E69F00","neck":"#009E73",
        "body":"#0072B2","wing":"#D55E00","leg":"#777777","tail":"#F0E442"}
COARSE_ORDER=["head","eye","beak","neck","body","wing","leg","tail"]
COLLAPSE_TOL=1e-8
pd.set_option("display.max_rows", 250)
pd.set_option("display.max_columns", 40)

def require(path,command):
    path=Path(path)
    if not path.exists(): raise FileNotFoundError(f"Missing {path}\nProduce it with: {command}")
    return path
def family(name): return str(name).split("::",1)[0]
def add_mapping(E):
    E=E.copy(); E["attribute_type"]=E.concept_name.map(family)
    E["mask_group"]=E.attribute_type.map(ATTRIBUTE_TYPE_TO_MASK); return E
def attach(E,V):
    local=add_mapping(E); V=V.rename(columns={"image_name":"image","coarse":"mask_group"})
    return local[local.mask_group.notna()].merge(
        V[["image","mask_group","pixel_count","area_frac","visible"]],
        on=["image","mask_group"],how="inner",validate="many_to_one")
def balanced_accuracy(y,pred):
    y=np.asarray(y).astype(int); pred=np.asarray(pred).astype(int)
    tpr=(pred[y==1]==1).mean() if (y==1).any() else np.nan
    tnr=(pred[y==0]==0).mean() if (y==0).any() else np.nan
    return np.nanmean([tpr,tnr])

VIS=require(CURATED/"cub70_visibility.parquet","bash data/cub70/prepare_all.sh")
E70P=require(CURATED/"cub70_eval"/"cub70-cbm-s1.parquet","CONFIGS='cub70-cbm' SEEDS='1' bash analysis/cub70_prepare_analysis.sh")
EFULLP=require(CURATED/"cub70_eval"/"cub-cbm-s1.parquet","CONFIGS='cub-cbm' SEEDS='1' bash analysis/cub70_prepare_analysis.sh")
RAWVIS=pd.read_parquet(VIS); V=coarse_visibility(RAWVIS,threshold=.001)
E70=add_mapping(pd.read_parquet(E70P)); EFULL=add_mapping(pd.read_parquet(EFULLP))
J70=attach(E70,V); JFULL=attach(EFULL,V)
identity_error=float(np.nanmax(np.abs(E70.prob.to_numpy()-1/(1+np.exp(-E70.z.clip(-50,50).to_numpy())))))
if identity_error>1e-5: raise RuntimeError(f"exported z is not the concept logit: max probability mismatch={identity_error}")
print(f"[EXPORTED RAW-LOGIT PASS] max |prob-sigmoid(z)|={identity_error:.3g}")
print("CUB70 rows:",len(E70),"images:",E70.image.nunique(),"species:",E70.y_true.nunique(),"concepts:",E70.concept_name.nunique())
print("mask-matched images:",J70.image.nunique(),"fine masks:",sorted(RAWVIS.part.unique()))


## 1 · What population and mask evidence are available?

**Question.** What population and mask evidence are available?

**Variables and prediction.** Count prediction images, mask-matched images, species, exact concepts, 11 released masks, and eight coarse groups. Coverage losses must be explicit before any visible-versus-hidden comparison.

**Method.** Report fine-mask visibility and bilateral left/right support without inventing left/right concepts. A mask is visible when area/image area is at least 0.001; the denominator is all 1,888 mask-matched photographs.

### Figure 1 · What population and mask evidence are available?

**How to read the figure.** Panel A shows, for each of the 11 released CUB masks, the fraction of 1,888
joined photographs where mask area is at least 0.001 of image area, the
declared visibility threshold.
Panel B shows the median visible mask area divided by image area. `leg` is the
CUB name; `foot` is never used here. Low coverage can mean true occlusion,
pose, or missing/coarse annotation, which later photographs must distinguish.
Visibility 0.25 means the released mask passes the threshold in 25% of the
1,888 joined photographs.


In [ ]:
# ALT: CUB70 inventory with visibility rates and median area for all 11 released part masks.
inventory=pd.DataFrame([
    {"population":"CUB70 prediction export","images":E70.image.nunique(),"species":E70.y_true.nunique(),"concepts":E70.concept_name.nunique()},
    {"population":"mask-matched CUB70","images":J70.image.nunique(),"species":J70.y_true.nunique(),"concepts":J70.concept_name.nunique()},
])
fine=RAWVIS.groupby("part").agg(images=("image_name","nunique"),visible_rate=("visible","mean"),median_area=("area_frac","median")).reindex(CUB70_PARTS)
display(inventory); display(fine.round(4))
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
axes[0].bar(fine.index,fine.visible_rate,color="#0072B2"); axes[0].tick_params(axis="x",rotation=55)
axes[0].set_ylabel("fraction of images with visible mask"); axes[0].set_title("A · Visibility of all 11 released masks")
axes[1].bar(fine.index,fine.median_area,color="#E69F00"); axes[1].tick_params(axis="x",rotation=55)
axes[1].set_ylabel("median mask area / image area"); axes[1].set_title("B · Visible-region size")
fig.suptitle("Figure 1 · CUB70 mask population and coverage")
plt.tight_layout(); plt.show()


### Review record for Figure 1

- **Literal observation:** The export contains 1,976 images, 70 species, and 112 concepts; the mask join retains 1,888 images, 67 species, and 107 concepts. Head/body/beak masks are present in over 92% of joined images, tail in 81%, bilateral wing/leg masks in about 60-65%, and eye/neck masks in only about 22-23%.
- **Strongest alternative explanation:** An absent released mask can mean missing/coarse annotation rather than physical occlusion.
- **Discriminating test:** Inspect bilateral counts, areas, and real images with all masks overlaid.
- **Limited conclusion:** `ACCEPTED FOR the stated CUB70 mask-analysis population and coverage limits; 88 images, three species, and five concepts are outside the mask-matched population.`
- **Next question:** Is a species/concept shortcut available before looking at model behavior?


## 2 · Is species–concept structure available before model behavior?

**Question.** Is species–concept structure available before model behavior?

**Variables and prediction.** For each exact selected concept, count supporting species, positive images, and the number of alternatives in its attribute type. Uneven support and species association make contextual prediction possible but do not prove model use.

**Method.** Use labels only; no model score appears in this figure.

### Figure 2 · Is species–concept structure available before model behavior?

**How to read the figure.** Each horizontal row is one exact concept. The x-axis is the number of the 70
species carrying that exact value. Dot color is the number of positive
photographs (yellow means more; purple means fewer), and dot area is the number of alternative
values in the same attribute type (larger means more alternatives). A gray
outline means no released-mask mapping. Example: x=20 means 20 species carry
that value. This is label structure, not model behavior.


In [ ]:
# ALT: Named CUB70 label-only plot showing species support, positive-image support, and number of alternatives for every exact concept; outlined dots mark concepts without a released-mask mapping.
LABEL=(E70.groupby(["attribute_type","concept_name","y_true"]).gt_label.mean().reset_index())
support=(LABEL.assign(supports=lambda d:d.gt_label>=.5).groupby(["attribute_type","concept_name"])
         .agg(species_support=("supports","sum"),species_total=("y_true","nunique")).reset_index())
pos=E70.groupby(["attribute_type","concept_name"]).gt_label.agg(positive_images="sum",total_images="size").reset_index()
support=support.merge(pos); support["alternatives_in_type"]=support.groupby("attribute_type").concept_name.transform("nunique")
support["mask_group"]=support.attribute_type.map(ATTRIBUTE_TYPE_TO_MASK)
support=support.sort_values(["attribute_type","concept_name"]).reset_index(drop=True)
y=np.arange(len(support)); fig,ax=plt.subplots(figsize=(12,max(16,.24*len(support))))
color_value=np.log1p(support.positive_images)
sizes=28+18*support.alternatives_in_type
edge=np.where(support.mask_group.isna(),"#555555","white")
sc=ax.scatter(support.species_support,y,c=color_value,s=sizes,cmap="viridis",
              edgecolors=edge,linewidths=.8)
ax.set_yticks(y); ax.set_yticklabels(support.concept_name,fontsize=7); ax.invert_yaxis()
ax.set_xlabel("number of CUB70 species carrying this exact concept value")
cb=fig.colorbar(sc,ax=ax,pad=.01)
cb.set_label("number of positive photographs (log color scale)")
from matplotlib.lines import Line2D
size_values=sorted(set([int(support.alternatives_in_type.min()),
                        int(support.alternatives_in_type.median()),
                        int(support.alternatives_in_type.max())]))
handles=[Line2D([0],[0],marker="o",linestyle="",markerfacecolor="#888888",
                markeredgecolor="white",markersize=np.sqrt(28+18*n)/1.5,
                label=f"{n} alternatives") for n in size_values]
handles.append(Line2D([0],[0],marker="o",linestyle="",markerfacecolor="#888888",
                      markeredgecolor="#555555",label="no released-mask mapping"))
ax.legend(handles=handles,loc="lower right",fontsize=8,title="dot size / outline")
fig.suptitle("Figure 2 · Exact-concept structure before model behavior")
plt.tight_layout(); plt.show(); display(support.round(3))


### Review record for Figure 2

- **Literal observation:** Exact values vary widely in supporting species and positive images, and attribute types contain one to six selected alternatives. Several size/shape concepts have no released-mask mapping.
- **Strongest alternative explanation:** Uneven label structure only makes a shortcut possible; it does not show model use.
- **Discriminating test:** Decode held-out species from the raw concept vector and individual part blocks.
- **Limited conclusion:** `ACCEPTED FOR uneven species/concept structure and an available contextual shortcut.`
- **Next question:** Does the learned representation actually store species information?


## 3 · How often is a positive label paired with no visible mapped region?

**Question.** How often is a positive label paired with no visible mapped region?

**Variables and prediction.** For concept `j`, conflict is `P(v_ig=0 | c_ij=1)`. High conflict means training/evaluation labels can be predicted without visible named-region evidence; it does not prove model use.

**Method.** Plot every exact mask-testable concept at a named y-position with its denominator.

### Figure 3 · How often is a positive label paired with no visible mapped region?

**How to read the figure.** Each named row is one exact concept. The x-value is a data fraction:
among images labelled positive for that concept, what fraction has no visible
mapped mask? It is not a predicted probability. A value of 0.8 means 80 of
every 100 positive-labelled examples lack a visible released mask.


In [ ]:
# ALT: Aligned named dot plot of positive-label/mask conflict rates and denominators for every testable CUB70 exact concept.
exact=[]
for (t,c),d in J70.groupby(["attribute_type","concept_name"]):
    pos=d[d.gt_label==1]; vis=pos[pos.visible]; hid=pos[~pos.visible]
    neg_hid=d[(d.gt_label==0)&(~d.visible)]
    exact.append({"attribute_type":t,"concept_name":c,"mask_group":d.mask_group.iloc[0],
                  "n_positive":len(pos),"n_visible":len(vis),"n_hidden":len(hid),
                  "label_mask_conflict":len(hid)/len(pos) if len(pos) else np.nan,
                  "z_visible":vis.z.mean() if len(vis) else np.nan,
                  "z_hidden":hid.z.mean() if len(hid) else np.nan,
                  "visibility_effect":vis.z.mean()-hid.z.mean() if len(vis) and len(hid) else np.nan,
                  "context_gap":hid.z.mean()-neg_hid.z.mean() if len(hid) and len(neg_hid) else np.nan,
                  "n_hidden_negative":len(neg_hid)})
# `support` carries a plotting-only mask_group column. Keep the
# row-level mask_group above instead of creating mask_group_x/y.
EXACT=pd.DataFrame(exact).merge(
    support.drop(columns=["mask_group"],errors="ignore"),
    on=["attribute_type","concept_name"],how="left"
)
EXACT=EXACT.sort_values(["attribute_type","concept_name"]).reset_index(drop=True)
y=np.arange(len(EXACT)); fig,ax=plt.subplots(figsize=(12,max(16,.24*len(EXACT))))
ax.scatter(EXACT.label_mask_conflict,y,c=EXACT.mask_group.map(COLORS).fillna("#BBBBBB"),s=24)
tick=[f"{r.concept_name}  (hidden/positive={int(r.n_hidden)}/{int(r.n_positive)})" for r in EXACT.itertuples()]
ax.set_yticks(y); ax.set_yticklabels(tick,fontsize=7); ax.invert_yaxis()
ax.set_xlim(-.02,1.02); ax.set_xlabel("fraction of positive labels with mapped mask absent")
ax.set_title("Figure 3 · Label/mask conflict for every exact testable concept")
plt.tight_layout(); plt.show(); display(EXACT[["concept_name","mask_group","n_positive","n_hidden","label_mask_conflict"]].round(3))


### Review record for Figure 3

- **Literal observation:** The positive-label/mask-absence fraction ranges from near zero to above 0.9. Body concepts are usually low, wing concepts are commonly about 0.13-0.28, and several throat/tail concepts are much higher.
- **Strongest alternative explanation:** Figure 12 shows that mask absence sometimes occurs even when the anatomical region is visibly present, especially for neck and beak mappings.
- **Discriminating test:** Inspect real photographs and all masks, then treat v=0 as released-mask absence rather than guaranteed physical occlusion.
- **Limited conclusion:** `ACCEPTED FOR label/released-mask conflict, not for an exact physical-occlusion rate.`
- **Next question:** Do positive-labelled raw scores differ when the mapped mask is present?


## 4 · Did the standard CUB70 CBM produce usable exact-concept outputs?

**Question.** Did the standard CUB70 CBM produce usable exact-concept outputs?

**Variables and prediction.** For every concept, compute raw-score spread, label separation, balanced accuracy, and positive recall. Exact collapse means `Q95(z)-Q05(z) <= 1e-8`; rounded probabilities are not used to diagnose collapse.

**Method.** Evaluate all 112 outputs and mark mask-testable concepts separately.

### Figure 4 · Did the standard CUB70 CBM produce usable exact-concept outputs?

**How to read the figure.** Each row is one of 112 exact concepts and the four aligned panels have the
same definitions as FunnyBird Figure 1: `spread=Q95(z)-Q05(z)`; `label
separation=median(z|c=1)-median(z|c=0)`; balanced accuracy averages positive
and negative recall; positive recall is `P(z>0|c=1)`. Zero spread within
`1e-8` is the declared collapse rule. Moving right is healthier for the last
three panels; spread only asks whether the output varies at all. For example,
label separation +2 means the positive-label median is two logit units above
the negative-label median.


In [ ]:
# ALT: Four aligned raw-score and thresholded-health plots for every CUB70 exact concept, with exact collapsed slots reported.
rows=[]
for (t,c),d in E70.groupby(["attribute_type","concept_name"]):
    pos=d[d.gt_label==1].z; neg=d[d.gt_label==0].z
    spread=np.quantile(d.z,.95)-np.quantile(d.z,.05)
    rows.append({"attribute_type":t,"concept_name":c,"mask_group":d.mask_group.iloc[0],
                 "spread":spread,"collapsed":spread<=COLLAPSE_TOL,
                 "label_separation":pos.median()-neg.median() if len(pos) and len(neg) else np.nan,
                 "balanced_accuracy":balanced_accuracy(d.gt_label,d.z>0),
                 "positive_recall":((pos>0).mean() if len(pos) else np.nan),
                 "n_positive":len(pos),"n_negative":len(neg)})
HEALTH=pd.DataFrame(rows).sort_values(["attribute_type","concept_name"]).reset_index(drop=True)
images=E70[["image","y_true","y_pred"]].drop_duplicates("image")
display(pd.DataFrame([{"images":len(images),"species":images.y_true.nunique(),
                      "task_accuracy":(images.y_true==images.y_pred).mean(),
                      "concept_accuracy":(E70.gt_label==E70.pred_label).mean()}]).round(4))
y=np.arange(len(HEALTH)); metrics=["spread","label_separation","balanced_accuracy","positive_recall"]
fig,axes=plt.subplots(1,4,figsize=(16,max(16,.24*len(HEALTH))),sharey=True)
colors=HEALTH.mask_group.map(COLORS).fillna("#BBBBBB")
for ax,m in zip(axes,metrics):
    ax.scatter(HEALTH[m],y,c=colors,s=17); ax.set_xlabel(m.replace("_"," "))
    if m=="label_separation": ax.axvline(0,color="black",lw=.8)
    if m in ["balanced_accuracy","positive_recall"]: ax.axvline(.5,color="gray",ls="--",lw=.8)
axes[0].set_yticks(y); axes[0].set_yticklabels(HEALTH.concept_name,fontsize=7); axes[0].invert_yaxis()
fig.suptitle("Figure 4 · Raw-score health guard for every exact CUB70 concept")
plt.tight_layout(); plt.show(); display(HEALTH[HEALTH.collapsed])
print("exact collapsed slots:",int(HEALTH.collapsed.sum()),"tolerance:",COLLAPSE_TOL)


### Review record for Figure 4

- **Literal observation:** Task accuracy is 0.1412 and concept accuracy 0.7105. Of 112 exact outputs, has_throat_color::grey is constant-positive and has_wing_pattern::multi-colored constant-negative; both have zero raw-z spread, zero label separation, and balanced accuracy 0.5. The other 110 outputs vary.
- **Strongest alternative explanation:** Low or uneven performance can reflect the CUB70 training setup and label noise; it does not by itself establish grounding failure.
- **Discriminating test:** Keep the two collapsed slots out of positive grounding claims and analyze all remaining slots with raw z.
- **Limited conclusion:** `ACCEPTED FOR 110 non-collapsed outputs; the two named collapsed outputs are unusable and remain explicit negative health results.`
- **Next question:** How often is a positive label paired with no released mapped mask?


## 4b · How much species identity is recoverable from the learned CUB70 concept vector?

**Question.** How much species identity is recoverable from the learned CUB70 concept vector?

**Variables and prediction.** Decode species from all raw concept logits and from each coarse mask-linked block on a held-out split. Accuracy above the 1/70 chance level shows that the learned representation stores species information; it does not prove that species caused a particular concept score.

**Method.** Build one image-by-concept matrix and use a fixed stratified 70/30 split.

### Figure 4b · How much species identity is recoverable from the learned CUB70 concept vector?

**How to read the figure.** The y-axis is held-out species accuracy. The reference line is 1/70 chance.
`all` uses all 112 raw concept logits; group bars use only logits mapped to
that CUB region. Above-chance values mean species identity is encoded, not
that species caused an individual concept prediction. Accuracy 0.14 is ten
times the 1/70 chance rate, but is still only 14% species accuracy.


In [ ]:
# ALT: Held-out CUB70 species-decoding accuracy from the complete raw concept vector and each coarse mask-linked concept block.
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
X=E70.pivot_table(index="image",columns="concept_name",values="z",aggfunc="first")
y=E70[["image","y_true"]].drop_duplicates().set_index("image").loc[X.index,"y_true"]
tr,te=train_test_split(np.arange(len(X)),test_size=.30,random_state=20260803,stratify=y)
cmap=E70[["concept_name","mask_group"]].drop_duplicates().set_index("concept_name").mask_group
blocks={"complete z":list(X.columns)}
blocks.update({g:[c for c in X.columns if cmap.get(c)==g] for g in COARSE_ORDER})
rows=[]
for name,cols in blocks.items():
    if not cols: continue
    model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    model.fit(X.iloc[tr][cols],y.iloc[tr]); rows.append({"block":name,"species_accuracy":accuracy_score(y.iloc[te],model.predict(X.iloc[te][cols])),"dimensions":len(cols)})
SPECIES_PROBE=pd.DataFrame(rows)
fig,ax=plt.subplots(figsize=(9,4)); ax.bar(SPECIES_PROBE.block,SPECIES_PROBE.species_accuracy,color=["#333333"]+[COLORS.get(x,"#BBBBBB") for x in SPECIES_PROBE.block.iloc[1:]])
ax.axhline(1/y.nunique(),color="black",ls="--",label="chance = 1/70"); ax.set_ylim(0,1)
ax.set_ylabel("held-out species accuracy"); ax.set_title("Figure 4b · Species decoded from CUB70 raw concept logits")
ax.legend(); plt.tight_layout(); plt.show(); display(SPECIES_PROBE.round(3))


### Review record for Figure 4b

- **Literal observation:** Species accuracy is 0.221 from all 112 logits versus 1/70 chance. Individual blocks are also above chance, led by wing 0.228, body 0.214, head 0.211, and tail 0.207.
- **Strongest alternative explanation:** Concepts naturally define bird species, so decodability is availability evidence, not proof that species caused any one score.
- **Discriminating test:** Hold exact concept and mask state fixed before estimating species differences.
- **Limited conclusion:** `ACCEPTED FOR species information in the CUB70 concept representation.`
- **Next question:** Are the exact concept outputs healthy enough to interpret?


## 5 · Does natural visibility change the raw score of a positive-labelled concept?

**Question.** Does natural visibility change the raw score of a positive-labelled concept?

**Variables and prediction.** `visibility_effect_j = mean(z|c=1,v=1)-mean(z|c=1,v=0)`. Positive values mean visible examples score higher; negative values require investigation rather than automatic backwash language.

**Method.** Require at least ten visible and ten hidden positive examples and show every eligible exact concept.

### Figure 5 · Does natural visibility change the raw score of a positive-labelled concept?

**How to read the figure.** Each named point is one exact concept. The x-axis is
`mean positive-labelled z when visible - mean positive-labelled z when hidden`.
Right of zero means visibility accompanies a higher raw score; left means the
visible group scores lower. Unlike a FunnyBird swap, these are different
photographs, so pose, species composition, and mask quality can also differ.


In [ ]:
# ALT: Zero-centered raw-logit visibility effects for every eligible CUB70 exact concept with visible and hidden counts.
VE=EXACT[(EXACT.n_visible>=10)&(EXACT.n_hidden>=10)&EXACT.visibility_effect.notna()].copy()
VE=VE.sort_values(["mask_group","attribute_type","concept_name"]).reset_index(drop=True)
y=np.arange(len(VE)); fig,ax=plt.subplots(figsize=(11,max(9,.23*len(VE))))
ax.scatter(VE.visibility_effect,y,c=VE.mask_group.map(COLORS).fillna("#BBBBBB"),s=30)
ax.axvline(0,color="black",lw=1); ax.set_yticks(y); ax.set_yticklabels(VE.concept_name,fontsize=6); ax.invert_yaxis()
ax.set_xlabel("visibility_effect in raw z units (visible − hidden)")
ax.set_title("Figure 5 · Natural-visibility effect for every eligible exact concept")
plt.tight_layout(); plt.show(); display(VE[["concept_name","mask_group","n_visible","n_hidden","z_hidden","z_visible","visibility_effect"]].round(3))


### Review record for Figure 5

- **Literal observation:** Across 48 eligible exact concepts, visibility_effect ranges from -0.917 to 1.124 raw-z units. Body and color concepts are often positive, while several bill, tail, and wing pattern/shape concepts are negative.
- **Strongest alternative explanation:** Visible and mask-absent photographs differ in species, pose, background, and mask quality; negative effects need not be inverse pixel use.
- **Discriminating test:** Test bilateral/area dose response, species matching, same-image model robustness, and real-image mask quality.
- **Limited conclusion:** `VALID OBSERVATIONAL TEST with mixed support: some concepts score higher with the mask present, but there is no universal CUB visibility response.`
- **Next question:** When the mapped mask is absent, does contextual label separation remain?


## 6 · Does contextual concept information remain when the named region is hidden?

**Question.** Does contextual concept information remain when the named region is hidden?

**Variables and prediction.** `context_gap_j = mean(z|c=1,v=0)-mean(z|c=0,v=0)`. A positive gap means outside-region information distinguishes the label while the mapped region is hidden; it is not a donor/source margin.

**Method.** Require at least ten hidden positives and ten hidden negatives.

### Figure 6 · Does contextual concept information remain when the named region is hidden?

**How to read the figure.** Each named point is one exact concept. The x-axis is the hidden-positive mean
raw score minus the hidden-negative mean raw score. A value of +4 means that,
even when the mapped region is absent, positive-labelled photographs score
four raw-logit units above negative-labelled photographs. That is contextual
prediction; it is not a donor/source margin and does not identify the cue.


In [ ]:
# ALT: Zero-centered raw-logit hidden-context gaps for every eligible CUB70 exact concept.
CG=EXACT[(EXACT.n_hidden>=10)&(EXACT.n_hidden_negative>=10)&EXACT.context_gap.notna()].copy()
CG=CG.sort_values(["mask_group","attribute_type","concept_name"]).reset_index(drop=True)
y=np.arange(len(CG)); fig,ax=plt.subplots(figsize=(11,max(9,.23*len(CG))))
ax.scatter(CG.context_gap,y,c=CG.mask_group.map(COLORS).fillna("#BBBBBB"),s=30)
ax.axvline(0,color="black",lw=1); ax.set_yticks(y); ax.set_yticklabels(CG.concept_name,fontsize=6); ax.invert_yaxis()
ax.set_xlabel("context_gap in raw z units (hidden positive − hidden negative)")
ax.set_title("Figure 6 · Hidden-region contextual separation")
plt.tight_layout(); plt.show(); display(CG[["concept_name","mask_group","n_hidden","n_hidden_negative","context_gap"]].round(3))


### Review record for Figure 6

- **Literal observation:** For 50 eligible exact concepts, context_gap is nonnegative and is positive for 48; it reaches 8.267 for yellow throat, 7.454 for buff throat, and above 4 for some tail patterns. The two zero gaps are the collapsed outputs.
- **Strongest alternative explanation:** Released-mask absence is a noisy proxy: species, pose, background, annotation quality, and visibly present but unmasked regions can all create separation.
- **Discriminating test:** Match species support, center within exact concept/mask state, and inspect the selected photographs and masks.
- **Limited conclusion:** `ACCEPTED FOR contextual label separation under released-mask absence; this is observational and is not a donor/source margin or causal CUB backwash proof.`
- **Next question:** Can bilateral visibility or region area explain the score patterns more simply?


## 7 · Do bilateral visibility and visible area offer simpler explanations?

**Question.** Do bilateral visibility and visible area offer simpler explanations?

**Variables and prediction.** For eye, wing, and leg, retain left/right masks and compare zero, one, or two visible sides. Separately estimate within-concept area dose response. A monotone increase supports local visual evidence; non-monotone patterns motivate pose or species controls.

**Method.** Use only positive-labelled rows and raw `z`.

### Figure 7 · Do bilateral visibility and visible area offer simpler explanations?

**How to read the figure.** The bilateral panel compares mean raw `z` when zero, one, or two eye/wing/leg
masks are visible. The area panel asks, within the same exact concept, whether
larger visible masks accompany higher `z`. A steady upward pattern would fit
local pixel reliance; mixed directions leave pose, species, and annotation as
alternatives. The area outcome is
`area_effect_j = mean(z | largest visible-area quartile, c=1) -
mean(z | smallest visible-area quartile, c=1)`; +1 means the largest-area
positive images score one raw-logit unit higher. Colors identify CUB groups.


In [ ]:
# ALT: CUB70 raw-logit response by number of visible bilateral masks and by within-concept visible-area quartiles.
pairmap={"eye":["left_eye","right_eye"],"wing":["left_wing","right_wing"],"leg":["left_leg","right_leg"]}
side=[]
for group,parts2 in pairmap.items():
    d=RAWVIS[RAWVIS.part.isin(parts2)]
    pv=d.pivot(index="image_name",columns="part",values="visible").fillna(False)
    pa=d.pivot(index="image_name",columns="part",values="area_frac").fillna(0)
    for image in pv.index:
        side.append({"image":image,"mask_group":group,"visible_sides":int(pv.loc[image].sum()),"bilateral_area":float(pa.loc[image].sum())})
SIDE=pd.DataFrame(side)
B=J70[(J70.gt_label==1)&J70.mask_group.isin(pairmap)].merge(SIDE,on=["image","mask_group"])
BS=B.groupby(["mask_group","visible_sides"]).agg(n=("z","size"),mean_z=("z","mean")).reset_index()
dose=[]
for (t,c),d in J70[(J70.gt_label==1)&(J70.area_frac>0)].groupby(["attribute_type","concept_name"]):
    if len(d)<20 or d.area_frac.nunique()<4: continue
    q=pd.qcut(d.area_frac,4,duplicates="drop")
    if q.nunique()<2: continue
    lo=d.loc[q==q.cat.categories[0],"z"].mean(); hi=d.loc[q==q.cat.categories[-1],"z"].mean()
    dose.append({"attribute_type":t,"concept_name":c,"mask_group":d.mask_group.iloc[0],"area_effect":hi-lo,"n":len(d)})
DOSE=pd.DataFrame(dose)
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
for g,d in BS.groupby("mask_group"): axes[0].plot(d.visible_sides,d.mean_z,"o-",label=g)
axes[0].set_xticks([0,1,2]); axes[0].set_xlabel("visible left/right masks"); axes[0].set_ylabel("mean raw z"); axes[0].legend()
for g,d in DOSE.groupby("mask_group"): axes[1].scatter([g]*len(d),d.area_effect,label=g,alpha=.65)
axes[1].axhline(0,color="black",lw=.8); axes[1].set_ylabel("largest-area quartile z − smallest-area quartile z")
fig.suptitle("Figure 7 · Bilateral visibility and area dose response")
plt.tight_layout(); plt.show(); display(BS.round(3)); display(DOSE.round(3))


### Review record for Figure 7

- **Literal observation:** Mean raw z is not monotone in zero/one/two visible sides for eye, leg, or wing. Within-concept area effects also span positive and negative values in every major group.
- **Strongest alternative explanation:** Species and pose composition can overwhelm a natural-image area comparison, and small masks may be missing rather than physically absent.
- **Discriminating test:** Hold exact concept and species fixed and evaluate held-out row-level prediction.
- **Limited conclusion:** `VALID TEST, NO SUPPORT for bilateral count or area as a sufficient universal explanation; local visual evidence may still matter for individual concepts.`
- **Next question:** Does performance differ by species after raw-label support is matched?


## 8 · Does concept performance differ between species after support is matched?

**Question.** Does concept performance differ between species after support is matched?

**Variables and prediction.** Join the original CUB per-image attribute labels to the CBM raw `z` predictions. For each exact concept, compare species that each contain at least three raw positive and three raw negative images. Equalize positive and negative support, then measure both recall gap and positive-row raw-z gap. Persistent gaps support species-dependent representation but remain observational.

**Method.** Use the refined CUB matching rule from `mcbm_recallv4`: raw image-level labels, deterministic vectorized bootstrap, at most 50 species pairs per exact concept, and explicit alignment/eligibility counts.

### Figure 8 · Does concept performance differ between species after support is matched?

**How to read the figure.** A matched pair contains two species with enough raw positive and negative
examples for the same exact concept. Positive/negative counts are equalized.
Both outcomes are absolute gaps, following the original recall notebooks.
One panel shows `|recall_A-recall_B|`; the companion shows
`|mean(z_pos,A)-mean(z_pos,B)|`. Zero means the matched species behave alike.
For each species pair, the bootstrap resamples positive images within each
species with replacement; the pair, not an individual image, is the summary
unit. A recall gap of 0.30 is a 30-percentage-point difference; a raw-z gap
of 2 is a two-logit-unit difference. These are health/species-dependence
diagnostics, not grounding proof.


In [ ]:
# ALT: Aligned CUB70 exact-concept plots of matched per-species positive-recall gaps and raw-logit gaps using original per-image CUB attribute labels; alignment and eligibility counts are displayed.
if "attribute_id" not in E70.columns:
    raise RuntimeError(
        "ERROR: CUB export lacks attribute_id; rerun cub70_export_eval.py "
        "after pulling the current repository"
    )
cub_root=CURATED/"CUB_200_2011"
raw_candidates=[cub_root/"attributes"/"image_attribute_labels.txt",
                cub_root/"image_attribute_labels.txt"]
raw_path=next((p for p in raw_candidates if p.exists()),None)
images_path=cub_root/"images.txt"
if raw_path is None or not images_path.exists():
    raise FileNotFoundError(
        f"ERROR: raw CUB annotations missing under {cub_root}; need "
        "image_attribute_labels.txt and images.txt"
    )
raw=pd.read_csv(raw_path,sep=r"\s+",header=None,usecols=[0,1,2,3])
raw.columns=["image_id","attribute_id","raw_label","certainty"]
raw=raw[raw.certainty>=1].drop_duplicates(["image_id","attribute_id"])
image_rows=[]
for line in images_path.read_text().splitlines():
    image_id,relative=line.split(maxsplit=1)
    image_rows.append({"image_id":int(image_id),"image":Path(relative).stem})
image_ids=pd.DataFrame(image_rows)
raw_eval=(E70.merge(image_ids,on="image",how="left",validate="many_to_one")
          .merge(raw[["image_id","attribute_id","raw_label","certainty"]],
                 on=["image_id","attribute_id"],how="inner",validate="one_to_one"))
alignment_rate=len(raw_eval)/len(E70)
if alignment_rate<0.98:
    raise RuntimeError(
        f"ERROR: raw-label alignment covered only {alignment_rate:.1%} of E70 rows"
    )
rng=np.random.default_rng(20260803); rows=[]; eligibility=[]; B=100
for (t,c),d in raw_eval.groupby(["attribute_type","concept_name"]):
    eligible=[]
    for sid,g in d.groupby("y_true"):
        pos=g[g.raw_label==1].z.to_numpy(); neg=g[g.raw_label==0].z.to_numpy()
        if len(pos)>=3 and len(neg)>=3:
            eligible.append((int(sid),pos,neg))
    eligibility.append({"attribute_type":t,"concept_name":c,
                        "eligible_species":len(eligible)})
    pairs=[(eligible[a],eligible[b]) for a in range(len(eligible)) for b in range(a+1,len(eligible))]
    if len(pairs)>50:
        pairs=[pairs[i] for i in rng.choice(len(pairs),50,replace=False)]
    for (sa,za,na),(sb,zb,nb) in pairs:
        mpos=min(len(za),len(zb)); mneg=min(len(na),len(nb))
        aa=za[rng.integers(len(za),size=(B,mpos))]
        bb=zb[rng.integers(len(zb),size=(B,mpos))]
        recall_gaps=np.abs((aa>0).mean(axis=1)-(bb>0).mean(axis=1))
        z_gaps=np.abs(aa.mean(axis=1)-bb.mean(axis=1))
        rows.append({"attribute_type":t,"concept_name":c,"species_a":sa,"species_b":sb,
                     "matched_positive_n":mpos,"matched_negative_n":mneg,
                     "recall_gap":recall_gaps.mean(),"recall_gap_lo":np.quantile(recall_gaps,.025),
                     "recall_gap_hi":np.quantile(recall_gaps,.975),
                     "raw_z_gap":z_gaps.mean(),"raw_z_gap_lo":np.quantile(z_gaps,.025),
                     "raw_z_gap_hi":np.quantile(z_gaps,.975)})
RECALL=pd.DataFrame(rows,columns=["attribute_type","concept_name","species_a","species_b",
    "matched_positive_n","matched_negative_n","recall_gap","recall_gap_lo","recall_gap_hi",
    "raw_z_gap","raw_z_gap_lo","raw_z_gap_hi"])
ELIGIBILITY=pd.DataFrame(eligibility)
if RECALL.empty:
    raise RuntimeError(
        "ERROR: raw image-level CUB labels produced no eligible matched species pairs"
    )
RS=(RECALL.groupby(["attribute_type","concept_name"]).agg(n_species_pairs=("recall_gap","size"),
     mean_recall_gap=("recall_gap","mean"),mean_raw_z_gap=("raw_z_gap","mean"),
     min_matched_positive_n=("matched_positive_n","min"),
     min_matched_negative_n=("matched_negative_n","min")).reset_index())
fig,axes=plt.subplots(1,2,figsize=(14,max(12,.24*len(RS))),sharey=True)
RS=RS.sort_values(["attribute_type","concept_name"]).reset_index(drop=True); y=np.arange(len(RS))
axes[0].scatter(RS.mean_recall_gap,y,c="#0072B2",s=24); axes[1].scatter(RS.mean_raw_z_gap,y,c="#E69F00",s=24)
axes[0].set_yticks(y); axes[0].set_yticklabels(RS.concept_name,fontsize=7); axes[0].invert_yaxis()
axes[0].set_xlabel("matched absolute positive-recall gap"); axes[1].set_xlabel("matched absolute positive-row raw-z gap")
fig.suptitle("Figure 8 · Species-matched concept differences")
plt.tight_layout(); plt.show()
display(pd.DataFrame([{"raw_alignment_rate":alignment_rate,"raw_rows":len(raw_eval),
                       "eligible_concepts":int((ELIGIBILITY.eligible_species>=2).sum()),
                       "matched_pairs":len(RECALL)}]).round(3))
display(RS.round(3))
display(RECALL.nlargest(25,"raw_z_gap")[["concept_name","species_a","species_b",
    "matched_positive_n","matched_negative_n","recall_gap","recall_gap_lo","recall_gap_hi",
    "raw_z_gap","raw_z_gap_lo","raw_z_gap_hi"]].round(3))


### Review record for Figure 8

- **Literal observation:** All 221,312 rows align to original per-image CUB labels; all 112 concepts yield eligible species and 5,190 matched pairs. Mean absolute positive-recall gaps reach about 0.53, and positive-row raw-z gaps range from zero to about 13.
- **Strongest alternative explanation:** Species still differ in pose, background, annotation certainty, and image quality; some matched supports are as small as three positives.
- **Discriminating test:** Replicate at the seed level and test species after exact concept and mask state with held-out images.
- **Limited conclusion:** `ACCEPTED FOR observational species-dependent concept performance after raw-label support matching, not for causal species backwash.`
- **Next question:** Do conflict, support, and alternatives organize the concept-level effects?


## 9 · Do conflict, support, and number of alternatives organize the exact-concept effects?

**Question.** Do conflict, support, and number of alternatives organize the exact-concept effects?

**Variables and prediction.** At the concept level, relate `visibility_effect` and `context_gap` to label/mask conflict, image support, species support, and alternatives in the attribute type. Held-out predictive improvement supports an organizing association, not a causal contribution.

**Method.** Use standardized numeric predictors and repeated five-fold ridge regression.

### Figure 9 · Do conflict, support, and number of alternatives organize the exact-concept effects?

**How to read the figure.** The y-axis is held-out RMSE for predicting either the exact-concept visibility
effect or context gap; lower is better. Starting from an intercept, conflict,
image support, species support, and number of alternatives are added. A drop
means the added concept-level information generalizes; a rise supplies no
explanatory credit. RMSE 1.2 to 1.0 is improvement; 1.2 to 1.3 is not.
This does not subtract causal effects.


In [ ]:
# ALT: Cross-validated concept-level error after sequentially adding label conflict, image support, species support, and number of alternatives.
from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
FEATURES=["label_mask_conflict","n_positive","species_support","alternatives_in_type"]
collapsed_names=set(HEALTH.loc[HEALTH.collapsed,"concept_name"])
ACCOUNT_BASE=EXACT[(EXACT.n_visible>=10)&(EXACT.n_hidden>=10)&(EXACT.n_hidden_negative>=10)
                   & ~EXACT.concept_name.isin(collapsed_names)].copy()
rows=[]
for outcome in ["visibility_effect","context_gap"]:
    d=ACCOUNT_BASE.dropna(subset=[outcome]).copy()
    cv=RepeatedKFold(n_splits=5,n_repeats=10,random_state=20260803)
    baseline=np.sqrt(np.mean((d[outcome]-d[outcome].mean())**2))
    for k in range(1,len(FEATURES)+1):
        model=make_pipeline(SimpleImputer(),StandardScaler(),Ridge(alpha=5.0))
        mse=-cross_val_score(model,d[FEATURES[:k]],d[outcome],cv=cv,scoring="neg_mean_squared_error")
        rows.append({"outcome":outcome,"stage":" + ".join(FEATURES[:k]),"rmse":float(np.sqrt(mse.mean())),"n_concepts":len(d)})
    rows.append({"outcome":outcome,"stage":"intercept only","rmse":baseline,"n_concepts":len(d)})
CONCEPT_ACCOUNT=pd.DataFrame(rows)
fig,axes=plt.subplots(1,2,figsize=(14,4.5))
for ax,outcome in zip(axes,["visibility_effect","context_gap"]):
    d=CONCEPT_ACCOUNT[CONCEPT_ACCOUNT.outcome==outcome]
    order=["intercept only"]+[" + ".join(FEATURES[:k]) for k in range(1,len(FEATURES)+1)]
    d=d.set_index("stage").reindex(order); ax.plot(range(len(d)),d.rmse,"o-")
    ax.set_xticks(range(len(d))); ax.set_xticklabels(["baseline","+ conflict","+ image support","+ species support","+ alternatives"],rotation=25,ha="right")
    ax.set_ylabel("cross-validated RMSE"); ax.set_title(outcome.replace("_"," "))
fig.suptitle("Figure 9 · Concept-level sequential observational accounting")
plt.tight_layout(); plt.show()
display(pd.DataFrame([{"shared_eligible_concepts":len(ACCOUNT_BASE),
    "excluded_collapsed":len(collapsed_names),"minimum_visible_positive":10,
    "minimum_hidden_positive":10,"minimum_hidden_negative":10}]))
display(CONCEPT_ACCOUNT.round(3))


### Review record for Figure 9

- **Literal observation:** The supplied render used different populations for the two outcomes (87 concepts for visibility_effect versus 48 for context_gap), so its RMSE curves are not a valid linked comparison of the contributor sequence. The revised cell fixes both outcomes to the same non-collapsed population with at least ten visible positives, ten hidden positives, and ten hidden negatives.
- **Strongest alternative explanation:** Changing eligibility can change both baselines and apparent predictor gains, so the old numerical comparison cannot be carried forward.
- **Discriminating test:** Rerender this single corrected shared-population analysis, then compare the two outcomes.
- **Limited conclusion:** `INCOMPLETE: code and population are corrected; rerendered Figure 9 must be inspected before assigning contributor credit.`
- **Next question:** Does species-dependent raw-z variation remain within concept and mask state?


## 10 · Does species explain raw-score variation within the same exact concept and visibility state?

**Question.** Does species explain raw-score variation within the same exact concept and visibility state?

**Variables and prediction.** First center `z` within each exact concept and visibility state, then summarize residual means by species. Persistent spread shows species-dependent contextual prediction beyond the current mask state.

**Method.** Require at least three rows for every displayed concept/state/species estimate.

### Figure 10 · Does species explain raw-score variation within the same exact concept and visibility state?

**How to read the figure.** First subtract the mean raw `z` for the same exact concept and visible/hidden
state. Each point then summarizes one species. Zero means the species matches
that controlled average; remaining spread means species still organizes the
score. Because photographs were not experimentally changed, this remains an
observational context effect. A residual of +3 means that species lies three
raw-logit units above the same concept-and-mask-state mean.


In [ ]:
# ALT: CUB70 species-level raw-logit residuals after centering within exact concept and visibility state for all eight coarse groups.
R=J70.copy(); R["concept_visibility_mean"]=R.groupby(["concept_name","visible"]).z.transform("mean")
R["z_after_concept_visibility"]=R.z-R.concept_visibility_mean
SP=(R.groupby(["mask_group","concept_name","visible","y_true"]).agg(n=("z","size"),residual=("z_after_concept_visibility","mean"))
      .reset_index().query("n>=3"))
fig,axes=plt.subplots(2,4,figsize=(16,8),sharey=True); axes=axes.ravel()
for ax,g in zip(axes,COARSE_ORDER):
    d=SP[SP.mask_group==g].sort_values("residual")
    ax.scatter(np.arange(len(d)),d.residual,s=12,color=COLORS[g],alpha=.7)
    ax.axhline(0,color="black",lw=.8); ax.set_title(f"{g}: {len(d)} estimates"); ax.set_xlabel("concept/state/species, sorted")
axes[0].set_ylabel("mean raw-z residual"); axes[4].set_ylabel("mean raw-z residual")
fig.suptitle("Figure 10 · Species variation after exact concept and mask state")
plt.tight_layout(); plt.show(); display(SP.groupby("mask_group").residual.agg(["min","median","max","std","count"]).round(3))


### Review record for Figure 10

- **Literal observation:** After centering within exact concept and mask state, species residuals retain wide ranges in all eight groups: approximately -31.6 to 10.2 for head, -24.2 to 8.6 for tail, and -20.5 to 10.5 for neck, with smaller but nonzero ranges elsewhere.
- **Strongest alternative explanation:** Small or uneven concept/state/species cells and correlated pose/background can produce extreme descriptive residuals.
- **Discriminating test:** Require held-out image prediction and shrunken estimates before giving species generalizing explanatory credit.
- **Limited conclusion:** `ACCEPTED FOR a descriptive species association after exact concept and mask state.`
- **Next question:** Does species reduce held-out row-level prediction error?


## 11 · What remains after row-level visibility and species are added sequentially?

**Question.** What remains after row-level visibility and species are added sequentially?

**Variables and prediction.** Predict raw `z` on stable held-out image folds: exact concept baseline, then mask visibility/area, then species. A reduction in held-out error shows organization by that block; remaining error is the residual, not proof of an unknown cause.

**Method.** Use training-fold shrunken group means and identical rows at every stage.

### Figure 11 · What remains after row-level visibility and species are added sequentially?

**How to read the figure.** The y-axis is held-out raw-`z` prediction error; lower is better. The same image
rows are used throughout. Start with exact concept identity, add mask
visibility and area, then add species. Each decrease measures extra predictive
organization on unseen images. The remaining nonzero error is the residual,
not automatically a new causal mechanism. RMSE 3.3 to 3.1 means the added
block improves unseen-image prediction by 0.2 logit units.


In [ ]:
# ALT: Held-out CUB70 raw-logit prediction error after sequentially adding visibility, area, and species to exact concept identity.
A=J70.copy(); A["area_bin"]=pd.qcut(A.area_frac,4,labels=False,duplicates="drop")
A["fold"]=A.image.map(lambda x:int(hashlib.sha1(str(x).encode()).hexdigest(),16)%5)
stages=[("exact concept",["concept_name"]),("+ visibility and area",["concept_name","visible","area_bin"]),
        ("+ species",["concept_name","visible","area_bin","y_true"])]
rows=[]
for stage,cols in stages:
    pred=pd.Series(index=A.index,dtype=float)
    for fold in range(5):
        tr=A[A.fold!=fold]; te=A[A.fold==fold]; prior=tr.z.mean()
        st=tr.groupby(cols).z.agg(["mean","count"]).reset_index(); st["estimate"]=(st["mean"]*st["count"]+prior*10)/(st["count"]+10)
        j=te[cols].merge(st[cols+["estimate"]],on=cols,how="left")
        pred.loc[te.index]=j.estimate.fillna(prior).to_numpy()
    rows.append({"stage":stage,"rmse":float(np.sqrt(np.mean((A.z-pred)**2))),"mae":float(np.mean(np.abs(A.z-pred)))})
ROW_ACCOUNT=pd.DataFrame(rows)
fig,ax=plt.subplots(figsize=(7,4)); ax.plot(ROW_ACCOUNT.stage,ROW_ACCOUNT.rmse,"o-",color="#0072B2")
ax.set_ylabel("held-out RMSE of raw z"); ax.set_title("Figure 11 · Row-level sequential observational accounting")
plt.tight_layout(); plt.show(); display(ROW_ACCOUNT.round(3))


### Review record for Figure 11

- **Literal observation:** Held-out raw-z RMSE changes from 3.285 with exact concept alone to 3.262 after visibility/area and 3.104 after species; MAE changes from 1.869 to 1.855 to 1.700.
- **Strongest alternative explanation:** Species can proxy pose, habitat, background, and collection effects, so predictive gain does not isolate a biological species-to-concept causal path.
- **Discriminating test:** A matched relabel/retrain or valid same-image intervention would be needed for a causal claim; neither is accepted for CUB yet.
- **Limited conclusion:** `ACCEPTED FOR generalizing contextual organization by species beyond exact concept and released-mask state; the causal source remains unresolved.`
- **Next question:** Do real images show true occlusion, missing masks, or pose artifacts at the extremes?


## 12 · Do the numerical extremes correspond to pose, coarse masks, collapse, or contextual prediction?

**Question.** Do the numerical extremes correspond to pose, coarse masks, collapse, or contextual prediction?

**Variables and prediction.** Select cases by declared numerical rules: high conflict/high context gap, high conflict/low gap, strong positive visibility effect, and negative visibility effect. The photograph and all 11 masks must be inspected before assigning an explanation.

**Method.** Display original image, complete mask overlay, exact variables, species, and sample counts.

### Figure 12 · Do the numerical extremes correspond to pose, coarse masks, collapse, or contextual prediction?

**How to read the figure.** Each case occupies two rows: a mapped-mask-absent positive image followed by
a mapped-mask-visible positive image for the same exact concept. Columns are
the photograph, the mapped-region overlay, and every available released-mask
overlay. Titles give species, exact concept, `c`, `c_hat`, raw `z`, and mapped
area; tables give the selection rule and denominators. The images decide
whether an extreme is genuine occlusion, missing/coarse annotation, or
plausible contextual prediction. Collapsed concepts are excluded.


In [ ]:
# ALT: Four rule-selected CUB70 cases, each showing hidden and visible photographs beside overlays of all available released masks and exact raw-logit records.
from PIL import Image
mask_root=CURATED/"cub70"/"masks"/"AnnotationMasksPerclass"
if not mask_root.is_dir(): mask_root=CURATED/"cub70"/"masks"
image_root=CURATED/"CUB_200_2011"/"images"; image_lookup={p.stem:p for p in image_root.rglob("*.jpg")}
collapsed_names=set(HEALTH.loc[HEALTH.collapsed,"concept_name"])
eligible=EXACT[(EXACT.n_visible>=10)&(EXACT.n_hidden>=10)&(EXACT.n_hidden_negative>=10)
    & EXACT.context_gap.notna()&EXACT.visibility_effect.notna()
    & ~EXACT.concept_name.isin(collapsed_names)].copy()
conflict_q75=float(eligible.label_mask_conflict.quantile(.75))
high=eligible[eligible.label_mask_conflict>=conflict_q75]
picks=[("high conflict + high context gap",high.nlargest(1,"context_gap").iloc[0]),
       ("high conflict + low context gap",high.nsmallest(1,"context_gap").iloc[0]),
       ("strong positive visibility effect",eligible.nlargest(1,"visibility_effect").iloc[0]),
       ("negative visibility effect",eligible.nsmallest(1,"visibility_effect").iloc[0])]
mask_colors={p:plt.cm.tab20(i/20) for i,p in enumerate(CUB70_PARTS)}
mapped_parts={"eye":["left_eye","right_eye"],"wing":["left_wing","right_wing"],
              "leg":["left_leg","right_leg"]}
def choose(row,state):
    d=J70[(J70.concept_name==row.concept_name)&(J70.gt_label==1)]
    d=d[d.visible] if state=="visible" else d[~d.visible]
    return d.iloc[(d.z-row.z_visible).abs().argmin()] if len(d) and state=="visible" else (d.iloc[(d.z-row.z_hidden).abs().argmin()] if len(d) else None)
def overlays(stem,group):
    rgb=np.asarray(Image.open(image_lookup[stem]).convert("RGB")); all_ov=rgb.astype(float)/255; mapped_ov=all_ov.copy()
    rr=RAWVIS[RAWVIS.image_name==stem]; cid=int(rr.class_idx.iloc[0])+1; present=[]
    for p in CUB70_PARTS:
        f=mask_root/str(cid)/f"{stem}_{p}.png"
        if not f.exists(): continue
        m=np.asarray(Image.open(f).convert("L"))>0
        if m.shape!=rgb.shape[:2]: m=np.asarray(Image.fromarray(m.astype("uint8")*255).resize((rgb.shape[1],rgb.shape[0]),Image.Resampling.NEAREST))>0
        all_ov[m]=.4*all_ov[m]+.6*np.array(mask_colors[p][:3]); present.append(p)
        if p in mapped_parts.get(group,[group]): mapped_ov[m]=.3*mapped_ov[m]+.7*np.array(mask_colors[p][:3])
    return rgb,mapped_ov,all_ov,present
# Two rows per case keep each photograph large enough to inspect in HTML:
# hidden original/mapped/all masks, then visible original/mapped/all masks.
records=[]; fig,axes=plt.subplots(8,3,figsize=(13,28))
for r,(label,row) in enumerate(picks):
    for offset,state in [(0,"hidden"),(1,"visible")]:
        rr=2*r+offset
        rec=choose(row,state)
        for k in range(3): axes[rr,k].axis("off")
        if rec is None: axes[rr,0].text(.5,.5,"no example",ha="center"); continue
        if rec.concept_name!=row.concept_name: raise RuntimeError("example/concept mismatch")
        rgb,mapped,all_ov,present=overlays(rec.image,row.mask_group)
        axes[rr,0].imshow(rgb); axes[rr,1].imshow(mapped); axes[rr,2].imshow(all_ov)
        axes[rr,0].set_title(f"case {r+1}: {label}\n{state}: {rec.image}; species {rec.y_true}\n{row.concept_name}\nc={int(rec.gt_label)}, c_hat={int(rec.pred_label)}, z={rec.z:.3f}, area={rec.area_frac:.4f}",fontsize=9)
        axes[rr,1].set_title(f"mapped {row.mask_group} mask",fontsize=10)
        axes[rr,2].set_title("all available masks\n"+", ".join(present),fontsize=8)
        records.append({"case":r+1,"rule":label,"state":state,"image":rec.image,"species":rec.y_true,
            "concept_name":row.concept_name,"mask_group":row.mask_group,"c":int(rec.gt_label),
            "c_hat":int(rec.pred_label),"z":rec.z,"area_frac":rec.area_frac,
            "label_mask_conflict":row.label_mask_conflict,"visibility_effect":row.visibility_effect,
            "context_gap":row.context_gap,"n_visible":row.n_visible,"n_hidden":row.n_hidden,
            "n_hidden_negative":row.n_hidden_negative})
fig.suptitle("Figure 12 · Rule-selected photographs and complete mask overlays")
plt.tight_layout(); plt.show()
display(pd.DataFrame([{"selection_rule":label,"conflict_q75_threshold":conflict_q75,**row.to_dict()} for label,row in picks])
    [["selection_rule","conflict_q75_threshold","concept_name","mask_group","label_mask_conflict","visibility_effect","context_gap","n_visible","n_hidden","n_hidden_negative"]].round(3))
display(pd.DataFrame(records).round(3))


### Review record for Figure 12

- **Literal observation:** The supplied render is defective: the high-conflict/high-gap statistic names has_throat_color::white, but its visible panel displays the collapsed has_throat_color::grey example. The old grid also omits the mapped-mask-only view and complete per-image records. The revised cell excludes collapsed concepts and asserts that every displayed record matches the selected exact concept.
- **Strongest alternative explanation:** Because the photograph/statistic join was wrong, that pair cannot distinguish physical occlusion from missing annotation.
- **Discriminating test:** Rerender the corrected six-panel-per-case audit and inspect every selected pair.
- **Limited conclusion:** `INCOMPLETE: corrected Figure 12 must be rendered and every photograph/mask pair inspected before a verdict.`
- **Next question:** Do context and visibility patterns transfer between CUB70 and full-CUB training?


## 12b · Do the main observational quantities depend entirely on training with only 70 species?

**Question.** Do the main observational quantities depend entirely on training with only 70 species?

**Variables and prediction.** On the same mask-matched photographs and exact concepts, compare CUB70-CBM and full-CUB-CBM visibility effects and context gaps. Agreement supports robustness to the training species population; disagreement limits transfer between the two models.

**Method.** Use identical definitions and plot only concepts measurable in both exports.

### Figure 12b · Do the main observational quantities depend entirely on training with only 70 species?

**How to read the figure.** Each point is the same exact concept measured on the same photograph population
by the CUB70-trained and full-CUB-trained CBMs after each model's raw logits
are standardized within exact concept. The diagonal means equal standardized
effect size. Agreement supports robustness to the training population;
scatter or sign changes mean the magnitude is not stable across models.
`(full=+0.5, CUB70=-0.5)` is a sign disagreement in within-concept standard-
deviation units.


In [ ]:
# ALT: Same-image comparison of raw-logit visibility effects and context gaps between CUB70-trained and full-CUB-trained CBMs.
def exact_effects(J):
    rows=[]
    for (t,c),d in J.groupby(["attribute_type","concept_name"]):
        scale=d.z.std(ddof=0)
        if not np.isfinite(scale) or scale<=COLLAPSE_TOL: continue
        d=d.copy(); d["z_standardized"]=(d.z-d.z.mean())/scale
        pos=d[d.gt_label==1]; vis=pos[pos.visible]; hid=pos[~pos.visible]; neg=d[(d.gt_label==0)&(~d.visible)]
        rows.append({"attribute_type":t,"concept_name":c,
                     "visibility_effect":vis.z_standardized.mean()-hid.z_standardized.mean() if len(vis)>=10 and len(hid)>=10 else np.nan,
                     "context_gap":hid.z_standardized.mean()-neg.z_standardized.mean() if len(hid)>=10 and len(neg)>=10 else np.nan})
    return pd.DataFrame(rows)
F70=exact_effects(J70); F=exact_effects(JFULL); P=F70.merge(F,on=["attribute_type","concept_name"],suffixes=("_cub70","_full"))
fig,axes=plt.subplots(1,2,figsize=(11,5))
for ax,m in zip(axes,["visibility_effect","context_gap"]):
    d=P.dropna(subset=[m+"_cub70",m+"_full"]); ax.scatter(d[m+"_full"],d[m+"_cub70"],s=25,alpha=.65)
    lo=min(d[m+"_full"].min(),d[m+"_cub70"].min()); hi=max(d[m+"_full"].max(),d[m+"_cub70"].max())
    ax.plot([lo,hi],[lo,hi],"k--",lw=.8); ax.axhline(0,color="gray",lw=.5); ax.axvline(0,color="gray",lw=.5)
    ax.set_xlabel("full-CUB CBM standardized "+m.replace("_"," ")); ax.set_ylabel("CUB70 CBM standardized "+m.replace("_"," ")); ax.set_title(f"{m.replace('_',' ')} (n={len(d)})")
fig.suptitle("Figure 12b · Same-image guard: CUB70-trained versus full-CUB-trained CBM")
plt.tight_layout(); plt.show()


### Review record for Figure 12b

- **Literal observation:** The supplied render puts unstandardized raw-logit effects from two separately trained models against an identity line. Since their logit scales differ, distance from that line has no valid magnitude interpretation. The revised cell standardizes raw z within exact concept separately for each model before computing effects.
- **Strongest alternative explanation:** Sign agreement in the old plot remains descriptive, but diagonal distance cannot support a transfer claim.
- **Discriminating test:** Rerender the standardized same-image comparison before judging effect-size stability.
- **Limited conclusion:** `INCOMPLETE: the standardized same-image Figure 12b must be rendered before judging robustness.`
- **Next question:** What can be concluded directly across FunnyBird and CUB?


## 13 · Direct question-matched FunnyBird/CUB evidence table

Figures 1–12b and the corresponding FunnyBird figures were displayed and
reviewed together on 2026-08-04.

| Scientific question | FunnyBird operation | CUB operation | Same operation? | Allowed conclusion |
|---|---|---|---|---|
| Are outputs usable? | all 26 healthy | 110/112 non-collapsed | yes | compare only healthy outputs |
| Do named pixels matter? | positive controlled `response_delta` for all parts | mixed natural `visibility_effect` | no | causal FunnyBird response; no universal CUB response |
| Does context remain? | source wins after donorward response | positive released-mask-absent `context_gap` | no | exact backwash predicate in FunnyBird; observational contextual separation in CUB |
| Does visibility contribute? | same-render target area improves margin | natural mask state/area/sides mixed | weaker in CUB | FunnyBird contributor accepted; CUB result is heterogeneous and mask-limited |
| Does exact value matter? | post-swap value confusion is strongly graded | natural exact-concept matching still leaves species gaps | no | value difficulty matters in FunnyBird; CUB has related observational variation |
| Does species matter? | descriptive residual remains, but held-out margin prediction does not improve | residual remains and species lowers held-out raw-z error | observational in both | CUB gives stronger generalizing association; neither is causal species manipulation |
| Do training labels cause part of it? | conflict measured; matched RLv2 belongs to notebook 03rl | no accepted CUB retraining | no | no causal label conclusion in either standard-CBM report |

### CUB causal boundary

Notebook 05 may conclude that CUB does or does not show converging
**observational ingredients** of context-dependent concept prediction. It
may not claim a CUB donor/source backwash event because no accepted donor
response exists.


## 14 · Standard-CUB evidence ledger

| Predicate or explanation | Direct measurement | Status after review |
|---|---|---|
| population and mask coverage understood | Figure 1 | `ACCEPTED WITH MISSING-MASK LIMIT` |
| species/concept shortcut available | Figure 2 | `ACCEPTED FOR AVAILABILITY` |
| label/released-mask conflict measured | Figure 3 | `ACCEPTED; NOT PHYSICAL-OCCLUSION RATE` |
| exact outputs usable | Figure 4 | `110 ACCEPTED; 2 COLLAPSED AND EXCLUDED FROM POSITIVE CLAIMS` |
| species information in learned representation | Figure 4b | `ACCEPTED FOR AVAILABILITY` |
| natural visibility effect | Figure 5 | `MIXED; NO UNIVERSAL RESPONSE` |
| hidden context separation | Figure 6 | `ACCEPTED OBSERVATIONALLY; NOT A DONOR/SOURCE MARGIN` |
| bilateral/area alternatives | Figure 7 | `VALID TEST, NO SUFFICIENT UNIVERSAL EXPLANATION` |
| matched recall and raw-z species gaps | Figure 8 | `ACCEPTED OBSERVATIONALLY` |
| concept-level accounting | Figure 9 | `INCOMPLETE: SHARED-ELIGIBILITY RERENDER/REVIEW REQUIRED` |
| species residual | Figure 10 | `DESCRIPTIVE ASSOCIATION` |
| row-level accounting | Figure 11 | `SPECIES LOWERS HELD-OUT ERROR` |
| visual explanations inspected | Figure 12 | `INCOMPLETE: CORRECTED GRID REQUIRES IMAGE-BY-IMAGE REVIEW` |
| same-image full-CUB robustness guard | Figure 12b | `INCOMPLETE: STANDARDIZED RERENDER/REVIEW REQUIRED` |

**Next report question.** Only after this ledger is reviewed may notebook
06 ask whether CUB MCBM changes the accepted observational quantities.


# Methods appendix · CUB edit proxies not used in the main claim

These completed attempts are preserved because they delimit what CUB's
available masks can support:

1. **Reciprocal whole-part deletion:** `METHOD NOT CALIBRATED FOR
   CROSS-DATASET CAUSAL COMPARISON`. The shared edit did not reproduce the
   clean FunnyBird deletion and sometimes damaged meaningful control regions.
2. **Randomized patch masking V1/V2:** selected examples supported local
   pixel response, but the all-part calibration and wing coverage were not
   sufficient for a population-level cross-dataset claim.
3. **Beak/tail paste pilot:** `VALID TEST, NO SUPPORT FOR POSITIVE DONOR
   RESPONSE`. Therefore its negative final margins cannot be interpreted as
   retained-source backwash.

These outcomes reject the proposed edit measurements for their intended
causal use. They do not reject the observational analyses in Figures 1–12
and do not weaken the validated FunnyBird renderer swap.

Full code and artifacts remain in the repository and `CURATED_DATA`; this
report does not rerun them.


# Provenance appendix

The table below records the live Git commit, prediction and mask inputs,
SHA-256 hashes, population counts, collapse tolerance, and exclusions.


In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""): h.update(block)
    return h.hexdigest()
commit=subprocess.run(["git","rev-parse","HEAD"],cwd=REPO,capture_output=True,text=True,check=True).stdout.strip()
prov=[]
for role,path in [("CUB70 prediction export",E70P),("full-CUB prediction export",EFULLP),("visibility parquet",VIS)]:
    prov.append({"role":role,"path":str(path),"sha256":sha256_file(path)})
display(pd.DataFrame(prov)); display(pd.DataFrame([{"git_commit":commit,"seed":1,"epoch":100,
    "prediction_images":E70.image.nunique(),"mask_matched_images":J70.image.nunique(),
    "species":E70.y_true.nunique(),"exact_concepts":E70.concept_name.nunique(),
    "collapsed_concepts":int(HEALTH.collapsed.sum()),"collapse_tolerance":COLLAPSE_TOL,
    "visibility_threshold_area_fraction":.001}]))
